# Day 076 | Intro to GANs

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning

**Author:** Ripon Chandra Malo

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Generator | Network that creates fake data from noise | Producing synthetic soil property vectors |
| Discriminator | Network that distinguishes real from fake | Quality control of generated samples |
| Adversarial training | Generator and discriminator compete | Arms race drives better generation |
| Mode collapse | Generator only produces limited variety | All generated soils look the same |
| Training stability | Balancing G and D learning rates | Preventing one network from dominating |

---

## Our Geotechnical Problem

A geotechnical database has **limited soil test results** from a site investigation.
We want to **augment the dataset** with realistic synthetic soil property vectors
that capture the multivariate distributions and correlations of real soil data.

We will:
1. Understand the Generator-Discriminator adversarial game
2. Build a simple GAN to generate realistic soil property vectors
3. Implement the training loop with proper loss functions
4. Discuss mode collapse and strategies to mitigate it

---

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## Part 1: The GAN Framework

A GAN consists of two networks playing a **minimax game**:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- **Generator G:** Takes random noise $z$ and produces fake data $G(z)$
- **Discriminator D:** Outputs probability that input is real

Think of it as a **counterfeiter** (G) vs **detective** (D):
- The counterfeiter gets better at making fake soil reports
- The detective gets better at spotting fakes
- Eventually, the fakes are indistinguishable from real data

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE REAL SOIL DATA (TRAINING SET)
# ───────────────────────────────────────────────────────

def generate_real_soil_data(n_samples=500, seed=42):
    """
    Generate multi-class soil data with correlations.
    
    Features: [moisture, dry_density, LL, PI, fines_pct, SPT_N, Cc, Su]
    """
    np.random.seed(seed)
    
    samples = []
    for _ in range(n_samples):
        # Random soil type (drives correlations)
        soil_type = np.random.choice([0, 1, 2], p=[0.4, 0.35, 0.25])
        
        if soil_type == 0:  # Clay
            w = np.random.normal(42, 8)
            gd = np.random.normal(14.5, 1.0)
            ll = np.random.normal(55, 10)
            pi = np.random.normal(25, 6)
            fines = np.random.normal(82, 6)
            spt = np.random.normal(6, 3)
            cc = np.random.normal(0.45, 0.12)
            su = np.random.normal(35, 12)
        elif soil_type == 1:  # Silt
            w = np.random.normal(28, 5)
            gd = np.random.normal(16.5, 0.8)
            ll = np.random.normal(32, 5)
            pi = np.random.normal(10, 3)
            fines = np.random.normal(60, 8)
            spt = np.random.normal(18, 6)
            cc = np.random.normal(0.18, 0.05)
            su = np.random.normal(75, 20)
        else:  # Sand
            w = np.random.normal(14, 3)
            gd = np.random.normal(18.0, 0.6)
            ll = np.random.normal(0, 1)  # non-plastic
            pi = np.random.normal(0, 0.5)
            fines = np.random.normal(8, 4)
            spt = np.random.normal(32, 8)
            cc = np.random.normal(0.04, 0.01)
            su = np.random.normal(0, 2)  # no undrained strength
        
        samples.append([w, gd, ll, pi, fines, spt, cc, su])
    
    data = np.array(samples)
    data = np.clip(data, 0, None)  # no negatives
    return data

feature_names = ['Moisture (%)', 'Dry Density', 'LL (%)', 'PI (%)',
                 'Fines (%)', 'SPT N', 'Cc', 'Su (kPa)']

real_data = generate_real_soil_data(n_samples=500)
print(f"Real data shape: {real_data.shape}")
print(f"Features: {len(feature_names)}")

In [ ]:
# ───────────────────────────────────────────────────────
# NORMALIZE AND PREPARE DATALOADER
# ───────────────────────────────────────────────────────

scaler = StandardScaler()
data_scaled = scaler.fit_transform(real_data)

X_real = torch.FloatTensor(data_scaled)
train_ds = TensorDataset(X_real)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

data_dim = real_data.shape[1]  # 8
noise_dim = 16  # latent noise dimension

print(f"Data dimension: {data_dim}")
print(f"Noise dimension: {noise_dim}")

## Part 2: Building the Generator and Discriminator

| Component | Input | Output | Goal |
|-----------|-------|--------|------|
| Generator | Random noise (16-dim) | Fake soil vector (8-dim) | Fool the discriminator |
| Discriminator | Soil vector (8-dim) | Probability real (scalar) | Correctly classify real vs fake |

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATOR AND DISCRIMINATOR NETWORKS
# ───────────────────────────────────────────────────────

class Generator(nn.Module):
    """Generates fake soil property vectors from random noise."""
    
    def __init__(self, noise_dim=16, data_dim=8, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """Classifies soil vectors as real or fake."""
    
    def __init__(self, data_dim=8, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        return self.net(x)


G = Generator(noise_dim=noise_dim, data_dim=data_dim)
D = Discriminator(data_dim=data_dim)

print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")
print(f"\nGenerator:\n{G}")
print(f"\nDiscriminator:\n{D}")

In [ ]:
# ───────────────────────────────────────────────────────
# TEST FORWARD PASS
# ───────────────────────────────────────────────────────

z_test = torch.randn(4, noise_dim)
fake_test = G(z_test)
d_real_test = D(X_real[:4])
d_fake_test = D(fake_test)

print(f"Noise input shape:          {z_test.shape}")
print(f"Generated fake shape:       {fake_test.shape}")
print(f"D(real) before training:    {d_real_test.squeeze().detach().numpy().round(3)}")
print(f"D(fake) before training:    {d_fake_test.squeeze().detach().numpy().round(3)}")

## Part 3: GAN Training Loop

The training alternates between updating D and G:

**Step 1 — Train Discriminator:**
- Maximize $\log D(x_{\text{real}}) + \log(1 - D(G(z)))$

**Step 2 — Train Generator:**
- Minimize $\log(1 - D(G(z)))$ (or equivalently, maximize $\log D(G(z))$)

---

In [ ]:
# ───────────────────────────────────────────────────────
# GAN TRAINING LOOP
# ───────────────────────────────────────────────────────

# Re-initialize for clean training
torch.manual_seed(42)
G = Generator(noise_dim=noise_dim, data_dim=data_dim)
D = Discriminator(data_dim=data_dim)

opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion = nn.BCELoss()

epochs = 500
d_losses = []
g_losses = []
d_real_scores = []
d_fake_scores = []

for epoch in range(epochs):
    epoch_d_loss = 0
    epoch_g_loss = 0
    epoch_d_real = 0
    epoch_d_fake = 0
    n_batches = 0
    
    for (real_batch,) in train_dl:
        batch_size = real_batch.shape[0]
        
        # Labels
        real_labels = torch.ones(batch_size, 1)
        fake_labels = torch.zeros(batch_size, 1)
        
        # ─── Train Discriminator ───
        D.train()
        
        # Real data
        d_real = D(real_batch)
        d_loss_real = criterion(d_real, real_labels)
        
        # Fake data
        z = torch.randn(batch_size, noise_dim)
        fake_data = G(z).detach()  # detach so G is not updated
        d_fake = D(fake_data)
        d_loss_fake = criterion(d_fake, fake_labels)
        
        d_loss = d_loss_real + d_loss_fake
        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()
        
        # ─── Train Generator ───
        G.train()
        
        z = torch.randn(batch_size, noise_dim)
        fake_data = G(z)
        d_fake_for_g = D(fake_data)
        g_loss = criterion(d_fake_for_g, real_labels)  # G wants D to say "real"
        
        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()
        
        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        epoch_d_real += d_real.mean().item()
        epoch_d_fake += d_fake.mean().item()
        n_batches += 1
    
    d_losses.append(epoch_d_loss / n_batches)
    g_losses.append(epoch_g_loss / n_batches)
    d_real_scores.append(epoch_d_real / n_batches)
    d_fake_scores.append(epoch_d_fake / n_batches)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}/{epochs} | D_loss: {d_losses[-1]:.4f}  G_loss: {g_losses[-1]:.4f}  "
              f"D(real): {d_real_scores[-1]:.3f}  D(fake): {d_fake_scores[-1]:.3f}")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING CURVES
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(d_losses, label='Discriminator', linewidth=1.5, alpha=0.8)
axes[0].plot(g_losses, label='Generator', linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('GAN Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(d_real_scores, label='D(real)', linewidth=1.5, color='#27ae60')
axes[1].plot(d_fake_scores, label='D(fake)', linewidth=1.5, color='#e74c3c')
axes[1].axhline(y=0.5, color='k', linestyle='--', alpha=0.3, label='Ideal equilibrium')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Discriminator Score')
axes[1].set_title('D Scores Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

fig.suptitle('GAN Training Progress  |  Soil Property Generation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Evaluating Generated Samples

A GAN is only useful if the generated data **looks realistic**.
We compare marginal distributions and correlations between real and generated.

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE SAMPLES AND COMPARE DISTRIBUTIONS
# ───────────────────────────────────────────────────────

G.eval()
n_gen = 500
with torch.no_grad():
    z = torch.randn(n_gen, noise_dim)
    fake_scaled = G(z).numpy()

# Inverse transform
fake_data = scaler.inverse_transform(fake_scaled)
fake_data = np.clip(fake_data, 0, None)

# Compare distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (ax, name) in enumerate(zip(axes, feature_names)):
    ax.hist(real_data[:, i], bins=25, alpha=0.6, color='#2980b9', density=True, label='Real')
    ax.hist(fake_data[:, i], bins=25, alpha=0.6, color='#e74c3c', density=True, label='Generated')
    ax.set_xlabel(name)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle('Real vs GAN-Generated Soil Properties', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# CORRELATION COMPARISON
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Real correlation matrix
corr_real = np.corrcoef(real_data.T)
im0 = axes[0].imshow(corr_real, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(feature_names)))
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_xticklabels([n[:8] for n in feature_names], rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels([n[:8] for n in feature_names], fontsize=8)
axes[0].set_title('Real Data Correlations')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Generated correlation matrix
corr_fake = np.corrcoef(fake_data.T)
im1 = axes[1].imshow(corr_fake, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(feature_names)))
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_xticklabels([n[:8] for n in feature_names], rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels([n[:8] for n in feature_names], fontsize=8)
axes[1].set_title('Generated Data Correlations')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

# Correlation error
corr_error = np.abs(corr_real - corr_fake).mean()
print(f"Mean absolute correlation difference: {corr_error:.4f}")

## Part 5: Mode Collapse

**Mode collapse** is a common GAN failure mode where the generator learns to produce
only a limited variety of outputs. For soil data, this would mean all generated
samples cluster around one soil type instead of representing the full range.

Signs of mode collapse:
- Generated distributions are too narrow
- Missing clusters (only generating one soil type)
- Low standard deviation in generated features

Mitigation strategies:
- Use Wasserstein loss instead of BCE
- Add noise to discriminator inputs (label smoothing)
- Train D less frequently than G

---

In [ ]:
# ───────────────────────────────────────────────────────
# MODE COLLAPSE ANALYSIS
# ───────────────────────────────────────────────────────

print("Diversity Comparison: Real vs Generated")
print("="*65)
print(f"{'Feature':<16} {'Real Std':>10} {'Gen Std':>10} {'Ratio':>10} {'Status':>10}")
print("-"*65)

for i, name in enumerate(feature_names):
    real_std = real_data[:, i].std()
    gen_std = fake_data[:, i].std()
    ratio = gen_std / real_std if real_std > 0 else 0
    status = 'OK' if 0.5 < ratio < 1.5 else 'WARNING'
    print(f"{name:<16} {real_std:>10.3f} {gen_std:>10.3f} {ratio:>10.3f} {status:>10}")

print("="*65)
print("Note: Ratio near 1.0 = good diversity. Low ratio = potential mode collapse.")

In [ ]:
# ───────────────────────────────────────────────────────
# SCATTER PLOTS: REAL vs GENERATED (KEY PAIRS)
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
pairs = [(0, 1), (0, 2), (0, 6), (4, 5), (2, 3), (5, 7)]

for ax, (i, j) in zip(axes.flatten(), pairs):
    ax.scatter(real_data[:, i], real_data[:, j], alpha=0.4, s=15, color='#2980b9', label='Real')
    ax.scatter(fake_data[:, i], fake_data[:, j], alpha=0.3, s=15, color='#e74c3c', label='Generated')
    ax.set_xlabel(feature_names[i])
    ax.set_ylabel(feature_names[j])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Real vs Generated  |  Property Scatter Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Practice Exercises

Try these exercises to deepen your understanding:

```python
# Exercise 1: Wasserstein GAN (WGAN)
# Replace BCE loss with Wasserstein distance.
# Remove sigmoid from discriminator (now called "critic").
# Add weight clipping: for p in D.parameters(): p.data.clamp_(-0.01, 0.01)

# Exercise 2: Label smoothing
# Replace real_labels = 1.0 with real_labels ~ Uniform(0.8, 1.0)
# This prevents the discriminator from becoming too confident.

# Exercise 3: Conditional GAN
# Add soil type as a condition to both G and D.
# This allows generating specific soil types on demand.

# Exercise 4: Training ratio
# Train D for 3 steps per 1 G step. Does this improve stability?
```

---

## Key Takeaways

1. **GANs use adversarial training** — a generator and discriminator compete to improve
2. **The generator learns** to produce realistic soil vectors that fool the discriminator
3. **Training is delicate** — loss curves oscillate rather than monotonically decrease
4. **Mode collapse** is a real risk where the generator produces limited variety
5. **Correlation preservation** is key for geotechnical applications

---

## How This Connects

| ML Concept | Geotechnical Application |
|-----------|------------------------|
| Generator | Creating synthetic soil test results for data augmentation |
| Discriminator | Quality control — are generated samples realistic? |
| Mode collapse | Generator only producing one soil type instead of full range |
| Correlation preservation | Ensuring generated profiles are physically consistent |
| Data augmentation | Expanding limited site investigation datasets |